In [2]:
import hashlib
import struct
import numpy as np

# ---------- 1) pin the question (provide your 64-byte block) ----------
def pack_words(words32):
    # words32: list of 16 unsigned 32-bit ints (W0..W15)
    return b"".join(struct.pack(">I", w & 0xffffffff) for w in words32)

# Example rails (edit W0..W15 to shape the question)
W = [
    0x14159265, 0x92653589, 0xebe6ad9a, 0x6d9aca76,   # phase + complement
    0x00000011, 0x00000013, 0xffffffee, 0xffffffec,   # twin primes + complements
    0x00000000, 0x00000000, 0x00000000, 0x00000000,   # torsion (E_perp lives here)
    0xdeadbeef, 0x01234567, 0x89abcdef, 0xfedcba98    # anchors
]
question_bytes = pack_words(W)  # 64 bytes

# ---------- 2) compile with SHA-256 ----------
digest = hashlib.sha256(question_bytes).digest()         # 32 bytes, 256 bits
rng = np.random.default_rng(int.from_bytes(digest, "big"))

# ---------- 3) map digest to a tiny MLP (no training) ----------
def make_mlp_from_digest(digest_bytes, in_dim=32, hid=32, out_dim=16):
    # use digest as seed to draw deterministic weights
    seed = int.from_bytes(digest_bytes, "big")
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 0.25, size=(hid, in_dim))
    b1 = rng.normal(0, 0.25, size=(hid,))
    W2 = rng.normal(0, 0.25, size=(out_dim, hid))
    b2 = rng.normal(0, 0.25, size=(out_dim,))
    return (W1, b1, W2, b2)

W1,b1,W2,b2 = make_mlp_from_digest(digest, in_dim=32, hid=32, out_dim=16)

def forward_once(x):
    h = np.tanh(W1 @ x + b1)
    y = W2 @ h + b2
    return y, h

# ---------- 4) provide a stimulus (typed, small) ----------
# e.g., 32-dim normalized feature vector (your substrate)
stimulus = np.linspace(-1, 1, 32)   # placeholder; replace with domain features
y, h = forward_once(stimulus)

# ---------- 5) meters (H, kappa, echo power, delta) ----------
def bit_harmony(vec):
    # normalize activations to pseudo-bits: sign as 0/1; target ~0.35 ones
    bits = (vec > 0).astype(np.int32)
    return bits.mean()

def echo_power(vec, bins=16):
    # proxy: variance of histogram of activations after tanh
    hist, _ = np.histogram(vec, bins=bins, range=(-1,1))
    m = hist.mean()
    return ((hist - m)**2).mean()

H = bit_harmony(h)                     # want ~0.35
Pi = echo_power(h)                     # want flat/low after alignment
delta = np.mean(np.abs(np.sort(h) - np.sort(h)))  # trivial δ here (idempotent check)

gate_ok = (abs(H - 0.35) < 0.05) and (Pi < 5.0)   # tighten empirically per backend
